# Empirical Bayes Estimation Deep Dive

This notebook explores how CAPO's Empirical Bayes procedure estimates the variance parameters from batch data.

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np

from capo.eb_core import (
    l_capo_fit_beta_and_weights,
    kband_weights,
    eb_statistics,
)

## L-CAPO: Estimating β

The L-CAPO algorithm estimates the length exponent β by solving:

$$\hat{\beta} = \arg\max_\beta \ell(\beta)$$

where $\ell(\beta)$ is the marginal log-likelihood.

In [ ]:
def generate_data(beta_true, n_traj=200, seed=42):
    """Generate synthetic data with known beta."""
    torch.manual_seed(seed)
    L = torch.randint(10, 300, (n_traj,)).float()
    # Variance ~ L^beta
    std = L ** (beta_true / 2) * 0.1
    g = torch.randn(n_traj) * std
    return g, L

# Test with different true beta values
true_betas = [0.0, 0.5, 1.0, 1.5]
estimated_betas = []

print("L-CAPO Beta Estimation:")
print("-" * 40)
for beta_true in true_betas:
    g, L = generate_data(beta_true, n_traj=500)
    beta_hat, w, m = l_capo_fit_beta_and_weights(g=g, L=L, max_iters=50)
    estimated_betas.append(beta_hat)
    print(f"True β = {beta_true:.2f}  →  Estimated β = {beta_hat:.3f}")

In [ ]:
# Visualize
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(true_betas, estimated_betas, s=100, zorder=5)
ax.plot([0, 2], [0, 2], 'k--', alpha=0.5, label='Perfect estimation')
ax.set_xlabel('True β', fontsize=12)
ax.set_ylabel('Estimated β', fontsize=12)
ax.set_title('L-CAPO Beta Recovery', fontsize=14)
ax.legend()
ax.set_xlim(-0.1, 1.6)
ax.set_ylim(-0.1, 1.6)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Effect of Sample Size

How does estimation quality depend on the number of trajectories?

In [ ]:
beta_true = 1.0
sample_sizes = [20, 50, 100, 200, 500, 1000]
n_trials = 10

results = {n: [] for n in sample_sizes}

for n in sample_sizes:
    for trial in range(n_trials):
        g, L = generate_data(beta_true, n_traj=n, seed=42 + trial)
        beta_hat, _, _ = l_capo_fit_beta_and_weights(g=g, L=L, max_iters=50)
        results[n].append(beta_hat)

# Plot
fig, ax = plt.subplots(figsize=(10, 5))

means = [np.mean(results[n]) for n in sample_sizes]
stds = [np.std(results[n]) for n in sample_sizes]

ax.errorbar(sample_sizes, means, yerr=stds, fmt='o-', capsize=5, capthick=2)
ax.axhline(y=beta_true, color='r', linestyle='--', label=f'True β = {beta_true}')
ax.set_xlabel('Number of Trajectories', fontsize=12)
ax.set_ylabel('Estimated β', fontsize=12)
ax.set_title('L-CAPO Estimation vs Sample Size', fontsize=14)
ax.set_xscale('log')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nKey insight: ~100-200 trajectories usually sufficient for stable estimation")

## Weight Visualization

Let's see how the estimated weights compare to oracle weights.

In [ ]:
# Generate data
beta_true = 0.8
g, L = generate_data(beta_true, n_traj=100, seed=123)

# Estimate beta and get weights
beta_hat, w_estimated, m = l_capo_fit_beta_and_weights(g=g, L=L, max_iters=50)

# Oracle weights (using true beta)
_, w_oracle = kband_weights(L=L, beta=beta_true, rho=0.0, eta=1.0, k=0)

# Equal weights (GRPO)
w_equal = torch.ones_like(L) / len(L)

print(f"True β = {beta_true:.2f}")
print(f"Estimated β = {beta_hat:.3f}")

# Sort by length for visualization
sort_idx = torch.argsort(L)
L_sorted = L[sort_idx]

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(L_sorted.numpy(), w_equal[sort_idx].numpy(), 'g--', label='Equal (GRPO)', alpha=0.7)
ax.plot(L_sorted.numpy(), w_oracle[sort_idx].numpy(), 'b-', label=f'Oracle (β={beta_true})', linewidth=2)
ax.plot(L_sorted.numpy(), w_estimated[sort_idx].numpy(), 'r--', label=f'L-CAPO (β̂={beta_hat:.2f})', linewidth=2)
ax.set_xlabel('Trajectory Length', fontsize=12)
ax.set_ylabel('Weight', fontsize=12)
ax.set_title('Weight Comparison: Oracle vs L-CAPO vs Equal', fontsize=14)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Summary

The L-CAPO algorithm:
1. Estimates β by maximizing the marginal likelihood
2. Works well with ~100+ trajectories
3. Produces weights close to the oracle (true β) weights
4. Is fast and suitable for online learning